## 1. Environment Setup and Experiment Import

This cell sets `KMP_DUPLICATE_LIB_OK` to avoid duplicate OpenMP runtime errors that can occur on some macOS/Intel environments. It then imports `experiment_mlp_gn_coverage` from `experiments.py`. This function builds the MLP objectives, runs the uniform discretisation baseline and/or the adaptive bundle method, and saves the resulting plot as a PNG file.

In [1]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from experiments import experiment_mlp_gn_coverage

## 2. Run the MLP Baseline and Generate a Plot

This cell runs the GN* coverage experiment for the non-convex MLP. It uses `K=3, p=10, n=20, h=8`, so the parameter dimension is `d = h*p + h + K*h + K = 115`. The value of `coarse_resolution=9/26/85` corresponds to `r=9/26/85` in the plot legend. Setting `run_baseline=True` and `run_adaptive=False` means that only the uniform discretisation baseline is plotted, without the adaptive bundle curve. The plot is saved as `mlp_K={K}_p={p}_n={n}_h={h}_err={err_label}(r={coarse_resolution}).png`.

In [8]:
K = 3
p = 10
n = 20
h = 8
seed = 10
coarse_resolution = 85
err_label = "1e-2"

result = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=15,
    steps_per_point_per_pass=50,
    eval_every_n_grads=5000,
    run_baseline=True,
    run_adaptive=False,
    out_path=f"mlp_K={K}_p={p}_n={n}_h={h}_err={err_label}(r={coarse_resolution}).png",
)

result["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=3, p=10, n=20, h=8, d=115  |  L=[3.613 3.739 2.503] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=6.6667e-01
  Baseline pass 1/15 | t=17.84s | iters=187050 | grad_evals=561150 | worst-case pc=1.1588e-03
  Baseline pass 2/15 | t=34.89s | iters=374100 | grad_evals=1122300 | worst-case pc=8.5768e-04
  Baseline pass 3/15 | t=51.39s | iters=561150 | grad_evals=1683450 | worst-case pc=4.4375e-04
  Baseline pass 4/15 | t=66.34s | iters=748200 | grad_evals=2244600 | worst-case pc=2.4299e-04
  Baseline pass 5/15 | t=82.67s | iters=935250 | grad_evals=2805750 | worst-case pc=1.7845e-04
  Baseline pass 6/15 | t=99.63s | iters=1122300 | grad_evals=3366900 | worst-case pc=1.4404e-04
  Baseline pass 7/15 | t=114.85s | iters=1309350 | grad_evals=3928050 | worst-case pc=1.3842e-04
  Baseline pass 8/15 | t=131.01s | iters=1496400 | grad_evals=4489200 | worst-case pc=1.4583e-04
  Baseline 

'mlp_K=3_p=10_n=20_h=8_err=1e-2(r=85).png'

## 3. Compare the Two Algorithms at K=4, p=15

This module fixes a single MLP problem: `K=4, p=15, n=20, h=8, seed=10`. It runs both algorithms on this same problem: the uniform discretisation baseline and the adaptive bundle method. The algorithm budget follows the default settings used to generate `mlp.png`; only the model parameters `K` and `p` are changed from `K=3, p=10` to `K=4, p=15`.

In [10]:
K = 4
p = 15
n = 20
h = 8
seed = 10

coarse_resolution = 26
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 5000
max_outer = 1000

comparison = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    max_outer=max_outer,
    lambda_selector="sample",
    lambda_random_starts=512,
    run_baseline=True,
    run_adaptive=True,
    out_path=f"mlp_compare_K{K}_p{p}_n{n}_h{h}(r={coarse_resolution})_baseline_vs_adaptive.png",
)

comparison["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=4, p=15, n=20, h=8, d=164  |  L=[4.407 2.072 3.034 4.41 ] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=7.5000e-01
  Baseline pass 1/15 | t=19.71s | iters=182700 | grad_evals=730800 | worst-case pc=4.1604e-03
  Baseline pass 2/15 | t=39.49s | iters=365400 | grad_evals=1461600 | worst-case pc=2.1927e-03
  Baseline pass 3/15 | t=61.44s | iters=548100 | grad_evals=2192400 | worst-case pc=1.7146e-03
  Baseline pass 4/15 | t=83.87s | iters=730800 | grad_evals=2923200 | worst-case pc=1.6077e-03
  Baseline pass 5/15 | t=105.72s | iters=913500 | grad_evals=3654000 | worst-case pc=1.5495e-03
  Baseline pass 6/15 | t=125.67s | iters=1096200 | grad_evals=4384800 | worst-case pc=1.4793e-03
  Baseline pass 7/15 | t=147.92s | iters=1278900 | grad_evals=5115600 | worst-case pc=1.4793e-03
  Baseline pass 8/15 | t=168.01s | iters=1461600 | grad_evals=5846400 | worst-case pc=1.4793e-03
  B

'mlp_compare_K4_p15_n20_h8(r=26)_baseline_vs_adaptive.png'

## 4. Increase the Adaptive Budget and Use the Baseline Final GN* as the Stopping Target

The plot in Section 3 shows that the baseline reaches a relatively low final GN*, while the adaptive method does not reach the baseline final GN* under the current parameters and budget. The red baseline curve ends at about `1.479e-03`, while the blue adaptive curve remains above that level within the `max_outer=1000` budget. This means the adaptive method stops because it hits the `max_outer` limit, not because it reaches the target.

If the baseline final error is very low, the adaptive method may require a much larger budget, and it may be difficult to reach that target under the current implementation or parameter choices. The conclusion should therefore be stated as: under the current adaptive parameters and budget, the adaptive method did not reach the baseline final GN*; increasing `max_outer` or adjusting the lambda-selection strategy, such as `lambda_selector` or `lambda_random_starts`, may be necessary.

The following experiment keeps the same `K=4, p=15, n=20, h=8, seed=10` problem and baseline settings, but increases the adaptive budget to `max_outer=5000, max_inner=50` and increases the number of lambda samples to `lambda_random_starts=2048`. Inside `experiment_mlp_gn_coverage`, the baseline final GN* is automatically used as the adaptive method's `target_cov`. The adaptive method stops early if it reaches this target; otherwise, it runs until `max_outer=5000`.

In [11]:
K = 4
p = 15
n = 20
h = 8
seed = 10

coarse_resolution = 26
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 5000

max_outer = 5000
max_inner = 50
lambda_random_starts = 2048

comparison_larger_adaptive_budget = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    max_outer=max_outer,
    max_inner=max_inner,
    lambda_selector="sample",
    lambda_random_starts=lambda_random_starts,
    run_baseline=True,
    run_adaptive=True,
    out_path=(
        f"mlp_compare_K{K}_p{p}_n{n}_h{h}"
        f"(r={coarse_resolution})_adaptive_outer{max_outer}_sample{lambda_random_starts}.png"
    ),
)

comparison_larger_adaptive_budget["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=4, p=15, n=20, h=8, d=164  |  L=[4.407 2.072 3.034 4.41 ] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=7.5000e-01
  Baseline pass 1/15 | t=20.10s | iters=182700 | grad_evals=730800 | worst-case pc=4.1604e-03
  Baseline pass 2/15 | t=40.47s | iters=365400 | grad_evals=1461600 | worst-case pc=2.1927e-03
  Baseline pass 3/15 | t=60.47s | iters=548100 | grad_evals=2192400 | worst-case pc=1.7146e-03
  Baseline pass 4/15 | t=80.67s | iters=730800 | grad_evals=2923200 | worst-case pc=1.6077e-03
  Baseline pass 5/15 | t=100.65s | iters=913500 | grad_evals=3654000 | worst-case pc=1.5495e-03
  Baseline pass 6/15 | t=120.35s | iters=1096200 | grad_evals=4384800 | worst-case pc=1.4793e-03
  Baseline pass 7/15 | t=140.21s | iters=1278900 | grad_evals=5115600 | worst-case pc=1.4793e-03
  Baseline pass 8/15 | t=160.06s | iters=1461600 | grad_evals=5846400 | worst-case pc=1.4793e-03
  B

'mlp_compare_K4_p15_n20_h8(r=26)_adaptive_outer5000_sample2048.png'

## 5. Compare K=4, p=15 Under a Similar Gradient Budget

Section 4 still uses `coarse_resolution=26`, which gives the baseline a gradient cost of about `10,962,000`, far above the adaptive method's budget. This section uses a fairer setup by reducing the baseline coarse grid to `coarse_resolution=10`, while keeping `n_passes=15` and `steps_per_point_per_pass=50`. For `K=4`, the baseline's total gradient cost is approximately `286 * 15 * 50 * 4 = 858,000`.

The adaptive method is set to `max_outer=4290, max_inner=50`, which corresponds to a maximum gradient cost of about `4290 * 50 * 4 = 858,000`. This makes the two methods comparable under a similar gradient budget, which is more suitable for studying algorithmic efficiency.

In [12]:
K = 4
p = 15
n = 20
h = 8
seed = 10

coarse_resolution = 10
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 5000

max_outer = 4290
max_inner = 50
lambda_random_starts = 2048

comparison_equal_gradient_budget = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    max_outer=max_outer,
    max_inner=max_inner,
    lambda_selector="sample",
    lambda_random_starts=lambda_random_starts,
    run_baseline=True,
    run_adaptive=True,
    out_path=(
        f"mlp_compare_K{K}_p{p}_n{n}_h{h}"
        f"(r={coarse_resolution})_equal_grad_budget.png"
    ),
)

comparison_equal_gradient_budget["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=4, p=15, n=20, h=8, d=164  |  L=[4.407 2.072 3.034 4.41 ] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=7.5000e-01
  Baseline pass 1/15 | t=1.62s | iters=14300 | grad_evals=57200 | worst-case pc=1.3789e-02
  Baseline pass 2/15 | t=3.23s | iters=28600 | grad_evals=114400 | worst-case pc=1.0098e-02
  Baseline pass 3/15 | t=4.85s | iters=42900 | grad_evals=171600 | worst-case pc=9.9998e-03
  Baseline pass 4/15 | t=6.44s | iters=57200 | grad_evals=228800 | worst-case pc=1.0000e-02
  Baseline pass 5/15 | t=8.07s | iters=71500 | grad_evals=286000 | worst-case pc=1.0000e-02
  Baseline pass 6/15 | t=9.66s | iters=85800 | grad_evals=343200 | worst-case pc=1.0000e-02
  Baseline pass 7/15 | t=11.32s | iters=100100 | grad_evals=400400 | worst-case pc=1.0000e-02
  Baseline pass 8/15 | t=12.93s | iters=114400 | grad_evals=457600 | worst-case pc=1.0000e-02
  Baseline pass 9/15 | t=14.56

'mlp_compare_K4_p15_n20_h8(r=10)_equal_grad_budget.png'

## 6. Increase K and p Further While Keeping the Section 5 Algorithm Parameters

This module further increases the model size from the Section 5 setting: `K=4, p=15` is changed to `K=5, p=20`, while `n=20, h=8, seed=10` are kept fixed. The algorithm parameters are also kept the same as in Section 5: `coarse_resolution=10, n_passes=15, steps_per_point_per_pass=50, max_outer=4290, max_inner=50, lambda_random_starts=2048`.

Note that even though the algorithm parameters are unchanged, increasing `K` increases the number of simplex grid points, so the baseline's actual gradient cost also increases. The adaptive method's gradient cost also scales proportionally with `K`. Therefore, this module is intended to observe algorithm behavior under a larger model size, not to keep the exact same gradient cost as in Section 5.

In [13]:
K = 5
p = 20
n = 20
h = 8
seed = 10

coarse_resolution = 10
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 5000

max_outer = 4290
max_inner = 50
lambda_random_starts = 2048

comparison_larger_kp_same_params = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    max_outer=max_outer,
    max_inner=max_inner,
    lambda_selector="sample",
    lambda_random_starts=lambda_random_starts,
    run_baseline=True,
    run_adaptive=True,
    out_path=(
        f"mlp_compare_K{K}_p{p}_n{n}_h{h}"
        f"(r={coarse_resolution})_same_algorithm_params.png"
    ),
)

comparison_larger_kp_same_params["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=5, p=20, n=20, h=8, d=213  |  L=[3.604 2.367 5.222 3.003 2.918] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=8.0000e-01
  Baseline pass 1/15 | t=6.91s | iters=50050 | grad_evals=250250 | worst-case pc=1.2613e-02
  Baseline pass 2/15 | t=13.85s | iters=100100 | grad_evals=500500 | worst-case pc=1.1699e-02
  Baseline pass 3/15 | t=20.82s | iters=150150 | grad_evals=750750 | worst-case pc=1.0471e-02
  Baseline pass 4/15 | t=27.81s | iters=200200 | grad_evals=1001000 | worst-case pc=1.0430e-02
  Baseline pass 5/15 | t=34.77s | iters=250250 | grad_evals=1251250 | worst-case pc=1.0420e-02
  Baseline pass 6/15 | t=41.81s | iters=300300 | grad_evals=1501500 | worst-case pc=1.0877e-02
  Baseline pass 7/15 | t=48.91s | iters=350350 | grad_evals=1751750 | worst-case pc=1.0989e-02
  Baseline pass 8/15 | t=55.91s | iters=400400 | grad_evals=2002000 | worst-case pc=1.0957e-02
  Baseli

'mlp_compare_K5_p20_n20_h8(r=10)_same_algorithm_params.png'